# 第 15 章 · Tensor Core：WMMA、mma.sync 与 ldmatrix

本章回答：Tensor Core 是什么，为什么比普通 CUDA core 快一个数量级？一个 warp 怎么"合作"做一个 16 × 16 × 16 的矩阵乘？WMMA 和 `mma.sync` 两套接口差在哪？`ldmatrix` 是干什么的？fragment（片段）里的数到底分在哪个线程手上？

配套讲义：仓库里 `cuda-programming/notes/lessons/15-tensor-core-mma.md`（讲解、动画、练习都在讲义里）。对应原文：PG §5.4.11 Warp Matrix Functions（`materials/text/cuda-programming-guide.txt` 第 36052–36420 行，类型 / 形状表在 §5.4.11.6 第 36318 行起）；PTX ISA 9.4 §9.7.16（`text/ptx-isa-9.4-focus-mma-wgmma-tcgen05-tma.txt`：§9.7.16.1 形状第 4583 行、§9.7.16.4 WMMA 第 4893 行、§9.7.16.5.8 m16n8k16 片段第 6693 行、§9.7.16.5.15 ldmatrix 第 8169 行）；Gordić《Inside NVIDIA GPUs》Tensor Cores 一节（`text/blog-gordic-inside-nvidia-gpus-matmul.txt` 第 745–800 行）；Oxford prac5

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 一条指令 | 16 × 8 × 16 | `M * N * K` | 2048 次乘加 |
| 第二步 WMMA 片段 | 256 / 32 | `16 * 16 // 32` | 8 个数 |
| 第三步 每个 lane 拿几个 | A 8、B 4、C/D 4 | `frag_a(lane)`、`frag_b`、`frag_cd` | 32 个 lane 合起来不重不漏：256、128、128 个位置 |
| 第三步 拆进 lane 再拼回 | $D = AB + C$ | `mma_sync(scatter(A, frag_a), …)` → `gather(…)` | 和 `A @ B + C` 完全一样 |
| 第五步 swizzle | 列号 XOR (行号 % 8) | `swizzle_chunk(r, 0)` | 8 行的第 0 块放到 0–7 八个不同位置 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `M`, `N`, `K`**（来自 `ch15_tensor_core_mma.py`）

In [ ]:
M, N, K = 16, 8, 16

**常量 `WARP`**（来自 `ch15_tensor_core_mma.py`）

In [ ]:
WARP = 32

**函数 `frag_a`**（来自 `ch15_tensor_core_mma.py`）：A（16 × 16，BF16）：lane 拿 8 个元素（4 个 32 位寄存器 × 2）。groupID = lane >> 2，t = lane % 4。

In [ ]:
def frag_a(lane: int) -> list[tuple[int, int]]:
    """A（16 × 16，BF16）：lane 拿 8 个元素（4 个 32 位寄存器 × 2）。groupID = lane >> 2，t = lane % 4。"""
    g, t = lane >> 2, lane % 4
    return [(g, 2 * t), (g, 2 * t + 1), (g + 8, 2 * t), (g + 8, 2 * t + 1),
            (g, 2 * t + 8), (g, 2 * t + 9), (g + 8, 2 * t + 8), (g + 8, 2 * t + 9)]

**函数 `frag_b`**（来自 `ch15_tensor_core_mma.py`）：B（16 × 8）：lane 拿 4 个元素（2 个 32 位寄存器）。

In [ ]:
def frag_b(lane: int) -> list[tuple[int, int]]:
    """B（16 × 8）：lane 拿 4 个元素（2 个 32 位寄存器）。"""
    g, t = lane >> 2, lane % 4
    return [(2 * t, g), (2 * t + 1, g), (2 * t + 8, g), (2 * t + 9, g)]

**函数 `frag_cd`**（来自 `ch15_tensor_core_mma.py`）：C / D（16 × 8，FP32）：lane 拿 4 个元素（4 个 32 位寄存器）。

In [ ]:
def frag_cd(lane: int) -> list[tuple[int, int]]:
    """C / D（16 × 8，FP32）：lane 拿 4 个元素（4 个 32 位寄存器）。"""
    g, t = lane >> 2, lane % 4
    return [(g, 2 * t), (g, 2 * t + 1), (g + 8, 2 * t), (g + 8, 2 * t + 1)]

**函数 `gather`**（来自 `ch15_tensor_core_mma.py`）：反过来：从 32 个 lane 的寄存器拼回小矩阵。

In [ ]:
def gather(regs: np.ndarray, frag, shape) -> np.ndarray:
    """反过来：从 32 个 lane 的寄存器拼回小矩阵。"""
    out = np.full(shape, np.nan)
    for lane in range(WARP):
        for v, (r, c) in zip(regs[lane], frag(lane)):
            out[r, c] = v
    return out

**函数 `scatter`**（来自 `ch15_tensor_core_mma.py`）：把一个小矩阵拆进 32 个 lane：返回形状 (32, 每 lane 元素数) 的"寄存器"表。

In [ ]:
def scatter(mat: np.ndarray, frag) -> np.ndarray:
    """把一个小矩阵拆进 32 个 lane：返回形状 (32, 每 lane 元素数) 的"寄存器"表。"""
    return np.array([[mat[r, c] for r, c in frag(lane)] for lane in range(WARP)])

**函数 `mma_sync`**（来自 `ch15_tensor_core_mma.py`）：模拟一条 mma.sync.m16n8k16：硬件把 32 个 lane 的片段合成矩阵，算 D = A × B + C，再按 D 的布局分回去。

In [ ]:
def mma_sync(a_regs, b_regs, c_regs) -> np.ndarray:
    """模拟一条 mma.sync.m16n8k16：硬件把 32 个 lane 的片段合成矩阵，算 D = A × B + C，再按 D 的布局分回去。"""
    A = gather(a_regs, frag_a, (M, K))
    B = gather(b_regs, frag_b, (K, N))
    C = gather(c_regs, frag_cd, (M, N))
    return scatter(A @ B + C, frag_cd)

**函数 `swizzle_chunk`**（来自 `ch15_tensor_core_mma.py`）：第 15 章第五步的 swizzle：以 16 字节为单位，物理位置 = 列号 XOR (行号 % 8)。

In [ ]:
def swizzle_chunk(row: int, chunk: int) -> int:
    """第 15 章第五步的 swizzle：以 16 字节为单位，物理位置 = 列号 XOR (行号 % 8)。"""
    return chunk ^ (row % 8)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 一条 m16n8k16 做多少次乘加")
print("  16 × 8 × 16 =", M * N * K)

In [ ]:
print("\n第二步 WMMA 16 × 16 的片段：每个线程", 16 * 16 // WARP, "个数")

In [ ]:
print("\n第三步 每个 lane 拿几个元素")
print(f"  A: {len(frag_a(0))} 个，B: {len(frag_b(0))} 个，C/D: {len(frag_cd(0))} 个")
for name, frag, shape in (("A", frag_a, (M, K)), ("B", frag_b, (K, N)), ("C/D", frag_cd, (M, N))):
    cover = {rc for lane in range(WARP) for rc in frag(lane)}
    print(f"  {name:<3}: 32 个 lane 合起来覆盖 {len(cover)} 个位置 = {shape[0]} × {shape[1]}，不重不漏")
print("  lane 5 拿 A 的：", frag_a(5))

In [ ]:
print("\n第三步 拆进 32 个 lane → mma → 拼回来")
rng = np.random.default_rng(0)
A = rng.integers(-3, 4, (M, K)).astype(np.float32)
B = rng.integers(-3, 4, (K, N)).astype(np.float32)
C = rng.integers(-3, 4, (M, N)).astype(np.float32)
d_regs = mma_sync(scatter(A, frag_a), scatter(B, frag_b), scatter(C, frag_cd))
print("  拼回来的 D 和 A @ B + C 完全一样：", np.array_equal(gather(d_regs, frag_cd, (M, N)), A @ B + C))
print("  lane 0 手里的 4 个 D 值：", d_regs[0])

In [ ]:
print("\n第五步 swizzle：8 行里逻辑上的第 0 块（16 字节）各放到第几块")
print("  ", [swizzle_chunk(r, 0) for r in range(8)])
print("  每一行仍是 0–7 的一个排列：", all(sorted(swizzle_chunk(r, c) for c in range(8)) == list(range(8))
                                      for r in range(8)))

## 3. 代码里才看得到的

1. **`lane >> 2` 和 `lane % 4`**：PTX 文档把 32 个 lane 分成 8 组、每组 4 个，groupID 决定行、组内编号决定列；WMMA 不公开这张表，`mma.sync` 公开。
2. **寄存器是 32 位的，一个装两个 BF16**：表里 A 的 8 个元素放在 4 个寄存器里，代码只模拟"谁拿哪些元素"，不模拟打包。
3. **用小整数做测试**：A、B、C 取 −3 到 3 的整数，乘加结果是精确的，所以能用 `np.array_equal` 而不是容差。

## 4. 自己改一改

- 打印 `frag_cd(lane)` 看 lane 0、1、4 各拿 D 的哪几个位置，和讲义说的"C 的布局和 A 的前 4 个一样"对一对。
- 故意把 `frag_b` 里的 `2 * t + 8` 改成 `2 * t + 2`，看"不重不漏"的检查哪里报错。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> Tensor Core 一条指令做一个小矩阵乘，一个 warp 一起干；
> 低精度进、FP32 累加，比 CUDA core 快一个数量级；
> fragment 把小矩阵拆进 32 个线程的寄存器：WMMA 不告诉你谁拿哪个，mma.sync 写在 PTX 里；
> ldmatrix 一条指令按布局搬进寄存器，swizzle 让它不撞 bank；
> 一代比一代大：warp → warpgroup → 单线程发起，同步 → 异步。